In [13]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

# Параметри
M, m, l, b, g = 1.0, 0.1, 0.5, 0.1, 9.81

def pendulum_eom(theta, theta_dot, F, x_dot):
    sin_th, cos_th = np.sin(theta), np.cos(theta)
    return (g * sin_th * (M + m) - (-F + m * l * theta_dot**2 * sin_th - b * x_dot) * cos_th) / (l * (M + m * sin_th**2))

def cart_eom(theta, theta_dot, F, x_dot):
    sin_th, cos_th = np.sin(theta), np.cos(theta)
    return (-F + m * l * theta_dot**2 * sin_th - m * g * sin_th * cos_th - b * x_dot) / (M + m * sin_th**2)

class PID:
    def __init__(self):
        # Потужніші коефіцієнти для швидкої стабілізації
        self.Kp, self.Kd = 60.0, 25.0 
    
    def get_force(self, state):
        x, x_dot, theta, theta_dot = state
        # Спрощений PD-регулятор: пряма стабілізація кута
        F = -(self.Kp * theta + self.Kd * theta_dot)
        return np.clip(F, -100, 100)

results = []
for th0 in [5.0, 12.0, 20.0]:
    pid = PID()
    def sys_rhs(t, state):
        F = pid.get_force(state)
        return [state[1], cart_eom(state[2], state[3], F, state[1]),
                state[3], pendulum_eom(state[2], state[3], F, state[1])]
    
    sol = solve_ivp(sys_rhs, (0, 5), [0.0, 0.0, np.radians(th0), 0.0], method='RK45')
    
    # Критерій: якщо кут в кінці менше 5 градусів - стабільна
    is_stable = np.abs(np.degrees(sol.y[2][-1])) < 5.0
    results.append({"Кут": th0, "Результат": "Стабільна" if is_stable else "Нестабільна"})

print(pd.DataFrame(results).to_string(index=False))

 Кут Результат
 5.0 Стабільна
12.0 Стабільна
20.0 Стабільна
